# BancoDeTodos — Exploración y elección de modelo (laboratorio)

Este cuaderno es la **fase de investigación**: aquí se entienden los datos y se prueba un modelo.

**Importante:** en el flujo MLOps, lo que va a “producción” no es el notebook en sí, sino los **scripts reproducibles** (`src/entrenar.py`) y los **artefactos** guardados en `artifacts/`.

Pasos sugeridos para el estudiante:
1. Ejecutar las celdas en orden.
2. Leer los comentarios: explican qué hace cada bloque.
3. Comparar con `src/entrenar.py`: el script debe reflejar las mismas columnas y un modelo coherente con lo aprendido aquí.

In [ ]:
# Importamos librerías estándar para análisis tabular y modelado.
from pathlib import Path

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

# Detectamos la raíz del proyecto: a veces Jupyter arranca en `notebooks/`, a veces en la raíz.
RAIZ = Path.cwd().resolve()
if not (RAIZ / "data" / "creditos_sinteticos.csv").exists():
    RAIZ = (RAIZ / "..").resolve()
RUTA_CSV = RAIZ / "data" / "creditos_sinteticos.csv"

print("Raíz del proyecto:", RAIZ.resolve())
print("¿Existe el CSV?", RUTA_CSV.exists())

Raíz del proyecto: D:\OneDrive\Desktop\Docencia Unicomfacauca Posgrados\MLops_dummy
¿Existe el CSV? True


In [ ]:




# Cargamos el archivo generado por src/generar_datos.py
df = pd.read_csv(RUTA_CSV)
df.head()

,ingreso_mensual,deuda_total,antiguedad_laboral_meses,num_creditos_activos,moras_ult_12m,incumplimiento
0,2867422.14,2439953.30,134,2,1,0
1,1565645.01,2064089.37,73,0,0,0
2,3504310.46,3134765.16,25,2,0,0
3,3817305.68,2349578.75,103,7,0,1
4,1039068.77,3516743.85,62,2,2,1


In [ ]:
# Estadísticas descriptivas: detectar escalas, valores extremos, etc.
df.describe()

,ingreso_mensual,deuda_total,antiguedad_laboral_meses,num_creditos_activos,moras_ult_12m,incumplimiento
count,2.500000e+03,2.500000e+03,2500.00000,2500.000000,2500.00000,2500.000000
mean,2.713269e+06,2.097475e+06,118.84920,3.468400,0.79840,0.428800
std,1.283687e+06,1.256197e+06,69.16738,2.293797,0.90049,0.495004
min,4.840849e+05,3.194260e+05,0.00000,0.000000,0.00000,0.000000
25%,1.814123e+06,1.240738e+06,59.00000,2.000000,0.00000,0.000000
50%,2.490624e+06,1.802182e+06,119.50000,3.000000,1.00000,0.000000
75%,3.297741e+06,2.570134e+06,178.00000,5.000000,1.00000,1.000000
max,1.045181e+07,1.203131e+07,239.00000,7.000000,5.00000,1.000000


In [ ]:
# Proporción de la clase positiva (incumplimiento).
# En problemas reales conviene mirar también el costo de equivocarse.
tasa = df["incumplimiento"].mean()
print(f"Proporción incumplimiento: {tasa:.2%}")

Proporción incumplimiento: 42.88%


In [ ]:
# Definimos X (entradas) e y (etiqueta). Los nombres deben coincidir con la app y con src/entrenar.py
COLUMNAS = [
    "ingreso_mensual",
    "deuda_total",
    "antiguedad_laboral_meses",
    "num_creditos_activos",
    "moras_ult_12m",
]
X = df[COLUMNAS]
y = df["incumplimiento"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print(X_train.shape, X_test.shape)

(1875, 5) (625, 5)


In [ ]:
# Modelo A: regresión logística (lineal, simple de explicar).
pipe_lr = Pipeline([
    ("escalado", StandardScaler()),
    ("modelo", LogisticRegression(max_iter=2000, random_state=42)),
])
pipe_lr.fit(X_train, y_train)
proba_lr = pipe_lr.predict_proba(X_test)[:, 1]
print("LogisticRegression AUC:", round(roc_auc_score(y_test, proba_lr), 3))

# Modelo B: bosque aleatorio (no lineal, un poco más flexible).
rf = RandomForestClassifier(
    n_estimators=80, max_depth=6, random_state=42, class_weight="balanced"
)
rf.fit(X_train, y_train)
proba_rf = rf.predict_proba(X_test)[:, 1]
print("RandomForest AUC:", round(roc_auc_score(y_test, proba_rf), 3))

# En este laboratorio el script oficial usa LogisticRegression para mantener el código corto.
# Como ejercicio: cambie src/entrenar.py para guardar el RandomForest y compare en Streamlit.

LogisticRegression AUC: 0.752
RandomForest AUC: 0.749


## Cierre

- Vuelva a la raíz del proyecto y ejecute `python -m src.entrenar` para **regenerar** `artifacts/`.
- Levante la app: `streamlit run app/streamlit_app.py`.

Así separa **experimentación** (notebook) de **empaquetado** (script + artefactos + app).